# 03 · Data Cleaning
**Goal:** Fix every data-quality problem `02_data_modeling.ipynb` found, so `04_data_integration.ipynb` can merge the four tables without silently producing wrong rows.

**Prerequisite:** read `02_data_modeling.ipynb` first — every task below is carried over from what that notebook found.

## Open decisions (not yet made — decide here, then implement)

- [ ] **Menu dedup rule.** Menu has 242 rows but only 106 unique `(product_name, size)` keys, and 24 of those groups have genuinely different `calories` / `price_usd` / `sugar_g` / `caffeine_mg` values (not harmless duplicate rows — see `02_data_modeling.ipynb`, the Caffè Latte Grande example ranges \$3.99–\$4.49 and 70–240 calories). Need to pick an aggregation rule to collapse each group to one row (e.g. median — more robust to outliers than mean given the spread) before Menu can be used as a dimension table.
- [ ] **2026-03 Macro gap.** Macro data ends `2026-02-01`; 122 transactions fall in `2026-03` with no matching macro month. Decide: leave `cpi`/`avg_hourly_earnings`/`real_wage_index` NULL for those rows, forward-fill the last known macro values, or drop the 122 rows.
- [ ] **Master dataset enrichment.** Decide whether to pull `weather.temp_max_f` / `weather.temp_min_f` and/or `macro.avg_hourly_earnings` / `macro.real_wage_index` into the master dataset (currently only `temp_f` and `cpi` are embedded in Transaction). This affects the join in `04_data_integration.ipynb`, so decide it here before writing that notebook's join code.

## Column-level nulls to resolve

From `01_data_audit.ipynb`'s `.isnull().sum()` output:

- [ ] `transactions.cpi` — 4,324 nulls. **Needs investigation, not just a fix**: this is far more than the 122 rows explained by the 2026-03 gap, so there's a second cause here.
- [ ] `transactions.total_price` — 4,324 nulls (same count as `cpi` — check whether it's the same rows before treating these as two separate problems).
- [ ] `transactions.caffeine_mg` — 8,186 nulls.
- [ ] `menu.caffeine_mg` — 23 nulls.
- [ ] `macro.cpi` / `macro.real_wage_index` — 1 null each.

## Output of this notebook

Cleaned versions of `menu` and `transactions` (plus `macro`/`weather` if anything turns up during the null investigation), ready to hand to `04_data_integration.ipynb` with no further fixes needed.

## Work starts below